In [1]:
Question={
    "office0":
    [
        ["Is this chair in this room?",'yes',8],
        ["Is this pillow in this room?",'no',-1]
    ],
    "office1":
    [
        ["Is this pillow in this room?",'yes',1298],
        ["Is this chair in this room?",'no',-1]
    ],
    "office2":
    [
        ["Is this table in this room?",'yes',339],
        ["Is this chair in this room?",'no',-1]
    ],
    "office3":
    [
        ["Is this table in this room?",'yes',1810],
        ["Is this chair in this room?",'no',-1]
    ],
    "office4":
    [
        ["Is this table in this room?",'yes',1744],
        ["Is this chair in this room?",'no',-1]
    ],
    "room0":
    [
        ["Is this sofa in this room?",'yes',1405],
        ["Is this chair in this room?",'no',-1]
    ],
    "room1":
    [
        ["Is this bed in this room?",'yes',433],
        ["Is this chair in this room?",'no',-1]
    ],
    "room2":
    [
        ["Is this table in this room?",'yes',1124],
        ["Is this chair in this room?",'no',-1]
    ],
}

In [8]:
import os
import cv2
from natsort import natsorted
import glob
import ast

def list_files_with_extension(dir_path, extension):
    '''
    列出指定目录下具有特定后缀的文件。
    
    :param dir_path: 要搜索的目录路径
    :param extension: 文件的后缀名（例如 '.txt'）
    :return: 一个包含符合条件的文件路径列表
    '''
    # 使用 glob 模块匹配通配符路径
    search_path = os.path.join(dir_path, f'*{extension}')
    files = natsorted(glob.glob(search_path))
    return files

def uniform_sample(lst, n):
    # 计算需要跳过的步长
    step = len(lst) / float(n)
    sampled_list = []
    # 使用步长进行均匀采样
    for i in range(n):
        index = int(i * step)
        sampled_list.append(lst[index])
    return sampled_list

def llm(a,b):
    prompt='Please check if the two words {} and {} are synonyms. If they are, return 1. If not, return 0,Please output the result directly'.format(a,b)

    completion = client.chat.completions.create(
        model="qwen-turbo",
        messages=[{"role": "user","content": [
            {"type": "text","text": prompt},
        ]}]
    )
    return int(completion.choices[0].message.content)

def acc_(output,label):
    right_1=0
    right_2=0
    right_3=0

    for idx,(i,j) in enumerate(zip(output,label)):
        if idx<10:
            if (i in [True,'yes'] and j in [True,'yes']) or (i in [False,'no'] and j in [False,'no']):
                right_1+=1
        
        if idx>=10 and idx<20:
            if int(i)==int(j):
                right_2+=1

        if idx>=20:
             if llm(i,j)==1:
                right_3+=1

    return right_1,right_2,right_3

import pickle
import base64
import json
import numpy as np

In [18]:
import os
from openai import OpenAI

client = OpenAI(
    # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx",
    api_key='', 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
import time

In [25]:
for idx,scene in enumerate(list(Question.keys())):
    
    #with open('/data/coding/eval/ablation/ours/3rscan_{}_3DSG.json'.format(idx+1),'r') as f:
    #    data=json.load(f)

    with open('/data/coding/eval_replica/output/'+scene+'/'+scene+'.json','r') as f:
        data=json.load(f)
    
    object_list=[]

    for i in data:
        object_list.append(i)


    info=[]

    for i in object_list:     ### 计算距离
        info.append({
            'id':i['id'],
            'label':i['description'],
            'best_view':i['color_image_idx'],
            #'max_distance':max_distance,
            #'min_distance':min_distance,
            #'max_label':max_label,
            #'min_label':min_label
        })

    label_list=[]         ## 唯一的label
    for i in info:
        if i['label'] not in label_list:
            label_list.append(i['label'])

    result={element: [] for element in label_list}
    for i in info:
        result[i['label']].append(i)

    process_prompt="This scene contains the following objects:{}".format(list(result.keys()))
    
    output_list=[]

    for iidx,j in enumerate(Question[scene]):   ### 遍历每个问题
        prompt="Please refer to the following data:{}, which is in the format of \
{{'id ': object index,\
'label', Object category,\
'best_view',the object's best view}},\
answer the following question:{},output the answer:,Do not output any other information, only give the values of 'yes' or' no 'and the possible' best view '".format(result,j[0])

        completion = client.chat.completions.create(
            model="qwen-max-2025-01-25", # 此处以qwen-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'user', 'content': prompt}],
            )
        time.sleep(1)
        print(completion.choices[0].message.content)
        output_list.append(completion.choices[0].message.content)

    #print(output_list)




yes 8
no
yes 12
yes 47
yes 124
yes 87
yes 264
yes 303
yes 94
yes 29
no
yes 388
yes 233
no
yes 53
yes 52


In [ ]:
response = client.chat.completions.create(
        model="glm-4v-plus-0111",  # 填写需要调用的模型名称
        messages=[
        {
            "role": "user",
            "content": [
            {
                "type": "img_url",
                "img_url": {
                    "url" : img_url
                }
            },
            {
                "type": "img_url",
                "img_url": {
                    "url" : img_url1
                }
            },
            {
                "type": "text",
                "text": "这两张图是否一样,是的话输出1,不是的话输出0"
            }
            ]
        }
        ]
    )

In [12]:
result

{'door': [{'id': 0, 'label': 'door', 'best_view': 374},
  {'id': 8, 'label': 'door', 'best_view': 132}],
 'v\n addCriterion\nv\n addCriterion\n': [{'id': 1,
   'label': 'v\n addCriterion\nv\n addCriterion\n',
   'best_view': 32},
  {'id': 2, 'label': 'v\n addCriterion\nv\n addCriterion\n', 'best_view': 13}],
 'jar': [{'id': 3, 'label': 'jar', 'best_view': 47}],
 'curtain': [{'id': 4, 'label': 'curtain', 'best_view': 71},
  {'id': 20, 'label': 'curtain', 'best_view': 77}],
 'table': [{'id': 5, 'label': 'table', 'best_view': 53},
  {'id': 14, 'label': 'table', 'best_view': 217},
  {'id': 22, 'label': 'table', 'best_view': 343},
  {'id': 31, 'label': 'table', 'best_view': 364}],
 'blinds': [{'id': 6, 'label': 'blinds', 'best_view': 23},
  {'id': 17, 'label': 'blinds', 'best_view': 319}],
 'shelf': [{'id': 7, 'label': 'shelf', 'best_view': 15}],
 'b\n addCriterion\nThe object is a small, green, leafy-like structure.': [{'id': 9,
   'label': 'b\n addCriterion\nThe object is a small, green, 